[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/07_traing_problem_and%20soultion/06_distributed_training/06_distributed_training.ipynb)

# 06. Distributed Training — DDP, FSDP & Parallelism

---


In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git
    os.chdir('Multimodal-Deep-Learning')
    os.chdir('07_traing_problem_and soultion/06_distributed_training')
    !pip install -q torch torchvision matplotlib numpy
else:
    nb_dir = os.getcwd()
    if not os.path.basename(nb_dir) == '06_distributed_training':
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '06_distributed_training'))
    sys.path.append(os.path.join(os.getcwd(), '..', '..'))

print(f'Working directory: {os.getcwd()}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})


## 1. Why Distributed?

$$
T \propto \frac{N_{steps} \cdot B_{local}}{G}
$$

$G$ = number of GPUs. Ideal speedup linear in $G$; communication adds overhead.


## 2. Data Parallel — AllReduce

$$
g = \frac{1}{G}\sum_{g=1}^{G} g_g
$$

Each GPU computes gradients on local batch; AllReduce averages them.

```
GPU0: g0 ──┐
GPU1: g1 ──┼── AllReduce ──> g = mean(g0,g1,...)
GPU2: g2 ──┘
```


## 3. Ring AllReduce Bandwidth

$$
\text{Data moved} = \frac{2(G-1)}{G} \cdot P \cdot \text{bytes}
$$

For $G=8$, factor $\approx 1.75$ — nearly optimal bandwidth utilization.


## 4. Model Parallelism

**Tensor parallel:** split $W$ across GPUs: $Y = X[W_1 \| W_2]$

**Pipeline parallel:** micro-batches reduce bubble ratio:

$$
\text{bubble} = \frac{G-1}{G-1+M}
$$

$M$ = number of micro-batches.


In [ ]:
def ring_allreduce_bytes(P, G, bytes_per=4):
    return 2 * (G - 1) / G * P * bytes_per

P = 7_000_000_000  # 7B params
for G in [2, 4, 8, 64]:
    gb = ring_allreduce_bytes(P, G) / 1e9
    print(f'G={G:2d}: AllReduce comm per step ≈ {gb:.2f} GB')


## 5. Practical Recipes

| GPUs | Recipe |
|------|--------|
| 1 | AMP + gradient checkpointing |
| 2–8 | DDP or FSDP |
| 8–64 | FSDP + tensor parallel |
| 64+ | 3D parallel (data + tensor + pipeline) |


In [ ]:
# DDP skeleton (runs on CPU for demo; use cuda + init_process_group in production)
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP

model = nn.Linear(10, 2)
print('DDP wraps model; each rank gets local batch, gradients AllReduced')
print('FSDP shards params + optimizer states across ranks (PyTorch 2.x)')

# Effective global batch
local_B, world_size = 8, 4
global_B = local_B * world_size
print(f'local_B={local_B}, world_size={world_size} -> global_B={global_B}')


In [ ]:
# Communication vs compute tradeoff
compute_ms = 200  # forward+backward per step
for G in [1, 2, 4, 8]:
    P = 1e9
    comm_ms = ring_allreduce_bytes(int(P), G) / 1e9 * 10  # rough: 10ms/GB
    eff = compute_ms / (compute_ms/G + comm_ms) if G > 1 else 1.0
    print(f'G={G}: comm~{comm_ms:.0f}ms, efficiency~{eff:.2f}x')


## 6. Pipeline Parallel — Bubble Visualization

```
Time ->
Stage0: [F0][F1][F2][F3]     (F=forward micro-batch)
Stage1:     [F0][F1][F2][F3]
Stage2:         [F0][F1][F2][F3]
Idle (bubble) at start/end reduced by increasing micro-batches M
```


## 7. FSDP vs DDP

| Feature | DDP | FSDP |
|---------|-----|------|
| Model replication | Full on each GPU | Sharded |
| Memory | Highest | Lowest |
| Communication | AllReduce grads | AllGather params |
| Best for | Fits in VRAM | Large models |


In [ ]:
# Pipeline bubble ratio
def bubble_ratio(stages, micro_batches):
    return (stages - 1) / (stages - 1 + micro_batches)

for M in [1, 4, 8, 16]:
    b = bubble_ratio(4, M)
    print(f'Stages=4, M={M:2d}: bubble={b:.1%}')


## 8. Multi-GPU NaN in Distributed Training

Distributed training **amplifies** numerical failures: larger effective batch without LR scaling → divergence; FSDP/ZeRO sharding → harder NaN localization; gradient checkpointing + AMP + FSDP → subtle wrapper ordering bugs.

This section covers DeepSpeed ZeRO, FSDP mixed precision, LR scaling, warmup scaling, and distributed debugging scenarios.


## 9. Learning Rate Linear Scaling Rule

When scaling from base batch $B_{\text{base}}$ to global batch $B_{\text{global}}$ across $G$ GPUs:

$$
\eta_{\text{new}} = \eta_{\text{base}} \times \frac{B_{\text{global}}}{B_{\text{base}}}
$$

**Example:** 4 GPUs × batch 32 = 128 global batch, base batch 32 → scale LR by **4×**.

Without this: effective batch grows but LR stays same → sharp loss spikes → NaN.

**When it breaks:** batch $> 8k$ → use LARS/LAMB (You et al. 2017/2020).


In [ ]:
# LR linear scaling calculator
def scaled_lr(base_lr, base_batch, num_gpus, local_batch):
    global_batch = num_gpus * local_batch
    return base_lr * (global_batch / base_batch)

base_lr, base_batch = 1e-4, 32
for gpus in [1, 2, 4, 8]:
    lr = scaled_lr(base_lr, base_batch, gpus, local_batch=32)
    print(f"{gpus} GPUs x 32 batch -> global={gpus*32:3d}, lr={lr:.6f} ({lr/base_lr:.0f}x base)")


## 10. Warmup MUST Scale with GPU Count

More GPUs → larger effective batch → optimizer sees larger gradient variance early.

Rule of thumb:

$$
\text{warmup\_steps} \propto G \quad \text{or} \quad \text{warmup\_steps} \propto \sqrt{G}
$$

Without adequate warmup: **NaN at steps 1–10** on multi-GPU is often misdiagnosed as "bad model."


In [ ]:
# Warmup schedules for different GPU counts
def warmup_lr(step, warmup_steps, base_lr, scaled_lr_val):
    if step < warmup_steps:
        return scaled_lr_val * (step + 1) / warmup_steps
    return scaled_lr_val

base_lr = 1e-4
local_batch = 32
steps = np.arange(0, 50)

fig, ax = plt.subplots(figsize=(10, 5))
for gpus in [1, 4, 8]:
    global_b = gpus * local_batch
    lr_scaled = base_lr * (global_b / 32)
    warmup = max(500, 100 * gpus)  # scale warmup with GPU count
    curve = [warmup_lr(s, warmup, base_lr, lr_scaled) for s in steps]
    ax.plot(steps, curve, label=f'{gpus} GPUs (warmup={warmup}, lr={lr_scaled:.2e})')

ax.set_xlabel('Step'); ax.set_ylabel('Learning rate')
ax.set_title('Warmup must grow with GPU count'); ax.legend(); plt.tight_layout(); plt.show()


## 11. DeepSpeed ZeRO NaN Debugging

ZeRO partitions optimizer states (Stage 1), gradients (Stage 2), parameters (Stage 3). NaN in one partition shard is harder to locate.

**Issues:**
- Gradient partitioning → partial NaN not visible on all ranks
- `GatheredParameters` context errors masking NaN source

**Fix:** use `deepspeed.utils.safe_get_full_grad` to inspect full gradients before step (when DeepSpeed installed).


In [ ]:
# DeepSpeed ZeRO NaN debugging patterns (fallback when DeepSpeed not installed)
try:
    import deepspeed
    from deepspeed.utils import safe_get_full_grad
    HAS_DS = True
except ImportError:
    HAS_DS = False

def inspect_grads_zero_style(model, rank=0):
    """ZeRO-style: gather and check full grad for NaN."""
    bad = []
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        g = p.grad
        if HAS_DS:
            try:
                g = safe_get_full_grad(p)
            except Exception:
                pass
        if torch.isnan(g).any() or torch.isinf(g).any():
            bad.append(name)
    if bad and rank == 0:
        print(f"NaN/Inf in {len(bad)} params: {bad[:3]}...")
    return bad

model = nn.Linear(10, 2)
model.weight.grad = torch.tensor([[1.0, float('nan')]])
bad = inspect_grads_zero_style(model)
print(f"DeepSpeed available: {HAS_DS}, bad params found: {len(bad)}")

print('ZeRO debugging checklist:')
print('  1. Enable grad clipping in DeepSpeed config: gradient_clipping=1.0')
print('  2. Use fp16/bf16 config with dynamic loss scaling (fp16) or bf16')
print('  3. Inspect with safe_get_full_grad before optimizer step')
print('  4. Stage 3: check GatheredParameters gather timing')


## 12. FSDP Mixed Precision — Safe vs Unsafe Combinations

FSDP casts parameters for forward compute. Wrong dtype combo → overflow in forward, NaN in backward.

**Recommended (safe):**

```python
from torch.distributed.fsdp import MixedPrecision
mp = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.float32,
    buffer_dtype=torch.float32,
)
```

| param_dtype | reduce_dtype | buffer_dtype | Verdict |
|-------------|--------------|--------------|---------|
| bfloat16 | float32 | float32 | Safe (recommended) |
| float16 | float32 | float32 | OK with GradScaler |
| float16 | float16 | float16 | Risky — NaN prone |
| bfloat16 | bfloat16 | float16 | Avoid |


In [ ]:
# FSDP MixedPrecision config examples (import-only demo on CPU)
try:
    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
    from torch.distributed.fsdp import MixedPrecision
    SAFE_MP = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.float32,
        buffer_dtype=torch.float32,
    )
    RISKY_MP = MixedPrecision(
        param_dtype=torch.float16,
        reduce_dtype=torch.float16,
        buffer_dtype=torch.float16,
    )
    print("Safe FSDP MixedPrecision:", SAFE_MP)
    print("Risky FSDP MixedPrecision:", RISKY_MP)
except ImportError:
    print("FSDP requires PyTorch 2.x — configs shown in markdown table above")

configs = [
    ("bf16/fp32/fp32", "Safe"),
    ("fp16/fp32/fp32", "OK with scaler"),
    ("fp16/fp16/fp16", "NaN prone"),
    ("bf16/bf16/fp16", "Avoid"),
]
print("\n".join(f"  {c[0]:20s} -> {c[1]}" for c in configs))


## 13. Gradient Checkpointing + FSDP + AMP — Wrapper Order

Each technique alone is stable; **combining all three** requires correct nesting:

```
CORRECT (outermost → innermost):
  FSDP(
    model with checkpointed blocks,
    mixed_precision=MixedPrecision(...),
  )
  + autocast in training loop
  + GradScaler only for fp16 (not bf16)

WRONG: checkpoint inside FSDP without use_reentrant=False (PyTorch 2.1+)
WRONG: autocast wrapping FSDP module incorrectly
```

Order matters because checkpoint recomputation must see the same dtype context as forward.


In [ ]:
# Correct wrapping order skeleton
WRAPPING_GUIDE = '''
# 1. Build model with gradient checkpointing on blocks
from torch.utils.checkpoint import checkpoint

class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.lin = nn.Linear(d, d)
    def forward(self, x):
        return F.relu(self.lin(x))

class Model(nn.Module):
    def __init__(self, d=64, n=4):
        super().__init__()
        self.blocks = nn.ModuleList([Block(d) for _ in range(n)])
    def forward(self, x):
        for blk in self.blocks:
            x = checkpoint(blk, x, use_reentrant=False)  # PyTorch 2.1+
        return x

# 2. Wrap with FSDP (outermost)
# model = FSDP(Model(), mixed_precision=SAFE_MP, device_id=rank)

# 3. Training loop: autocast + scaler outside forward
# with autocast(dtype=torch.bfloat16):
#     loss = model(batch)
# scaler.scale(loss).backward()  # fp16 only
'''
print(WRAPPING_GUIDE)


## 14. DDP + AMP Full Training Loop (Multi-GPU Safe Pattern)

Production pattern combining: linear LR scaling, warmup, grad accumulation, sentinel, clipping.


In [ ]:
def ddp_safe_step(model, optimizer, scaler, batch, accum_steps, step_in_accum,
                    sentinel, max_norm=1.0, use_amp=True):
    """One micro-step of DDP-safe AMP training."""
    with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.bfloat16):
        loss = model(batch).mean() / accum_steps

    if torch.isnan(loss) or torch.isinf(loss):
        return None, True

    scaler.scale(loss).backward()
    is_last = (step_in_accum + 1) % accum_steps == 0
    if is_last:
        scaler.unscale_(optimizer)
        sentinel.sanitize(model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    return loss.item() * accum_steps, False

print("Use with: DistributedDataParallel(model), DistributedSampler, linear LR scaling")


## 15. Distributed NaN Debugging Scenarios

Four real-world cases mirroring notebook 05 — with distributed-specific fixes.


### Scenario A: CLIP on 4 GPUs — NaN at Step 500

Same root cause as notebook 05 Scenario 1, plus **missing 4× LR scale** if global batch increased. Fix: clamp τ, BF16, grad clip, scale LR by GPU count.


In [ ]:
# Distributed CLIP fix checklist
fixes = {
    "clamp_tau": "tau = log_tau.exp().clamp(min=0.01)",
    "bf16": "autocast(dtype=torch.bfloat16)",
    "grad_clip": "clip_grad_norm_(model.parameters(), 1.0)",
    "lr_scale": "lr = base_lr * num_gpus * local_batch / base_batch",
    "warmup": "warmup_steps = 500 * num_gpus",
}
for k, v in fixes.items():
    print(f"  {k:12s}: {v}")


### Scenario B: 8 GPUs, NaN After 10k Steps — Loss Scale = 1

Monitor `scaler.get_scale()` across ranks. When scale hits 1, FP16 is one overflow away from death. Migrate to BF16 + weight decay.


In [ ]:
# Track loss scale across simulated 8-GPU run
scales = []
monitor = LossScaleMonitor(init_scale=65536.0)
grad_norms = list(np.linspace(0.5, 50, 200))  # creeping norms
for step, gn in enumerate(grad_norms):
    had_nan = gn > 40
    s = monitor.step(had_nan)
    scales.append(s)
    if s <= 1.0:
        print(f"Step {step}: loss scale hit {s:.1f} — migrate to BF16 NOW")
        break

plt.plot(scales); plt.title('Loss scale decay before late NaN'); plt.xlabel('Step'); plt.show()


### Scenario C: Rank 3 NaN First — Data Poisoning via AllReduce

Use `DistributedDataParallel` with `find_unused_parameters=False` only when safe. Always validate local batch before forward on **every** rank.


In [ ]:
# Rank-aware batch validation before DDP forward
class DistributedDataGuard:
    def __init__(self, rank):
        self.rank = rank
        self.validator = DataValidator(rank)

    def safe_forward(self, model, batch):
        if not self.validator.validate(batch):
            return None
        out = model(batch)
        if torch.isnan(out).any():
            print(f"[rank {self.rank}] NaN in forward output — skip backward")
            return None
        return out

# Reuse DataValidator from notebook 05 toolkit pattern
class DataValidator:
    def __init__(self, rank=0):
        self.rank = rank
        self.skipped = 0
    def validate(self, batch):
        if torch.isnan(batch).any() or torch.isinf(batch).any():
            self.skipped += 1
            return False
        return True

guard = DistributedDataGuard(rank=3)
batch = torch.tensor([[1.0, float('nan')]])
result = guard.safe_forward(nn.Linear(2, 2), batch)
print(f"Corrupted batch blocked: {result is None}")


### Scenario D: LLaVA + LoRA + FSDP on 4 GPUs

Safe config table for multimodal finetuning without projector NaN.


In [ ]:
LLAVA_FSDP_SAFE = {
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_scaling": "alpha/r = 1.0",
    "precision": "bfloat16",
    "fsdp_mp": "MixedPrecision(bf16, fp32 reduce, fp32 buffer)",
    "grad_clip": 1.0,
    "lr": "2e-5 * num_gpus (with warmup)",
    "warmup_ratio": 0.03,
    "checkpoint": "use_reentrant=False on projector blocks only",
}
for k, v in LLAVA_FSDP_SAFE.items():
    print(f"{k:18s}: {v}")


## 16. LARS / LAMB — When Linear Scaling Breaks

For batch sizes above ~8k (common at 64+ GPUs), plain SGD + linear scaling diverges.

**LARS** (You et al. 2017): layer-wise adaptive scaling for large batch.

**LAMB** (You et al. 2020): Adam-like trust ratio, used in BERT pretraining at batch 64k.

Switch when: NaN persists despite correct LR scaling + warmup + BF16.


In [ ]:
# When to switch from linear scaling to LARS/LAMB
global_batches = [128, 512, 2048, 8192, 32768]
for gb in global_batches:
    rec = "Linear scaling + warmup" if gb < 8192 else "LARS/LAMB required"
    print(f"global_batch={gb:5d} -> {rec}")


## 17. Megatron-LM & 3D Parallelism NaN Notes

Tensor + pipeline + data parallel (Shoeybi et al. 2019, Narayanan et al. 2021):

- Tensor parallel splits attention softmax across ranks → need **vocab-parallel cross-entropy** (stable log-sum-exp)
- Pipeline bubbles don't cause NaN but mask **which stage** produced NaN — log micro-batch ID + stage rank
- Always use **loss scaling** or BF16 in Megatron-style training


## 18. Multi-GPU NaN Debugging Checklist (Distributed)

```
Distributed NaN?
├── Scaled LR with GPU count? (eta * global_batch / base_batch)
├── Warmup scaled with G?
├── FSDP MixedPrecision safe combo?
├── Grad checkpoint use_reentrant=False?
├── ZeRO stage: safe_get_full_grad check?
├── All ranks same code path? (no rank-0-only collectives)
├── NCCL_DEBUG=INFO for hangs
└── Per-rank grad norm logging before AllReduce
```


## 19. Environment Variables for Multi-GPU Debugging

```bash
export NCCL_DEBUG=INFO
export NCCL_ASYNC_ERROR_HANDLING=1
export TORCH_NCCL_BLOCKING_WAIT=1
export TORCH_DISTRIBUTED_DEBUG=DETAIL
export CUDA_LAUNCH_BLOCKING=1   # slow but pinpoints NaN kernel
```


In [ ]:
# Launch helper (print recommended env vars)
import os
DEBUG_ENV = {
    "NCCL_DEBUG": "INFO",
    "NCCL_ASYNC_ERROR_HANDLING": "1",
    "TORCH_NCCL_BLOCKING_WAIT": "1",
    "TORCH_DISTRIBUTED_DEBUG": "DETAIL",
}
print("Recommended debug env (set before torchrun):")
for k, v in DEBUG_ENV.items():
    print(f"  export {k}={v}")


## 20. Updated Scaling Recipe Table (with NaN Prevention)

| GPUs | Recipe | NaN Prevention |
|------|--------|----------------|
| 1 | AMP + checkpointing | BF16, grad clip |
| 2–8 | DDP + linear LR scale | Warmup ∝ G, sentinel |
| 2–8 large model | FSDP + safe MixedPrecision | bf16/fp32/fp32 |
| 8–64 | FSDP + tensor parallel | LARS if batch > 8k |
| 64+ | 3D parallel + DeepSpeed | ZeRO grad inspect, stable CE |


## 21. Distributed NaN Prevention Recipes

| Scenario | NaN Mechanism | Distributed Fix |
|----------|---------------|-----------------|
| DDP AllReduce | One rank NaN → all ranks NaN | GradientSentinel before sync; validate per-rank batch |
| FSDP sharding | NaN hidden in param shard | MixedPrecision(bf16, fp32, fp32); inspect full grad |
| ZeRO Stage 3 | Partial NaN in grad partition | `safe_get_full_grad`; grad clipping in DeepSpeed config |
| LR not scaled | Effective batch ↑, LR same → spike | $\eta_{new} = \eta_{base} \times B_{global}/B_{base}$ |
| Short warmup | Large global batch, tiny warmup → NaN step 1–10 | warmup_steps $\propto G$ |
| Grad accum + AMP | Unscale inside micro-loop | Unscale **once** after $K$ accumulation steps |
| BatchNorm DDP | $B_{local} < 4$ → zero variance | SyncBatchNorm or GroupNorm/LayerNorm |
| Pipeline parallel | NaN in stage 3, hard to locate | Log micro-batch ID + stage rank; per-stage NaN hooks |
| Tensor parallel | Vocab-parallel CE overflow | Stable log-sum-exp cross-entropy (Megatron-style) |
| NCCL hang | Rank desync (not NaN but related) | Same collectives all ranks; NCCL_DEBUG=INFO |

### Launch Checklist (copy before every multi-GPU run)

```bash
# 1. Scale LR
# lr = base_lr * num_gpus * local_batch / base_batch

# 2. Scale warmup
# warmup_steps = max(500, 100 * num_gpus)

# 3. Debug env (remove in production)
export NCCL_DEBUG=INFO
export TORCH_DISTRIBUTED_DEBUG=DETAIL

# 4. Precision
# autocast(dtype=torch.bfloat16)  # NOT float16 on A100+

# 5. Safety
# clip_grad_norm_(model.parameters(), 1.0)
# GradientSentinel.sanitize(model) before optimizer.step()
```


## 22. Comprehensive NaN Prevention Table (All Model Types × Distributed)

| Model Type | Single-GPU NaN Source | Distributed Amplifier | Full Prevention Recipe |
|------------|----------------------|----------------------|------------------------|
| CLIP / Contrastive | τ collapse, log(0) | FP16 overflow × AllReduce | Clamp τ≥0.01, BF16, grad clip 1.0, LR scale × G |
| Transformer LM | Attention FP16 overflow | Long seq × tensor parallel | Pre-LN, flash attention, BF16, vocab-parallel CE |
| Image Captioning | log(0) in CE | Empty caption on one rank | Label smoothing, min len=1, DataValidator per rank |
| VQA | Soft CE zero probs | — | log_softmax, clamp probs, sentinel |
| LoRA Finetuning | Large α/r | FSDP dtype mismatch | α=r, MixedPrecision(bf16/fp32/fp32), clip 1.0 |
| Diffusion | σ schedule edge | Pipeline stage NaN | Clamp σ∈[1e-4, 1], per-stage hooks |
| GAN | −log(D(G(z))) | D stronger on one GPU | WGAN-GP, spectral norm, sync D/G updates |
| Multi-GPU DDP | Any local NaN | AllReduce poisoning | Sentinel + validate before backward |
| FSDP | Mixed precision | Wrong reduce dtype | bf16/fp32/fp32 only |
| DeepSpeed ZeRO | Sharded NaN | Hidden in partition | safe_get_full_grad, gradient_clipping=1.0 |

See notebook **05** Section 0 for IEEE 754 fundamentals, 15 NaN sources, `detect_anomaly()`, and `NaNTracer`.


## References & Further Reading

### Papers
- Micikevicius et al. (2018) — Mixed Precision Training — [arXiv:1710.03740](https://arxiv.org/abs/1710.03740)
- Ott et al. (2019) — fairseq: FP16 Training — [arXiv:1904.10509](https://arxiv.org/abs/1904.10509)
- Rajbhandari et al. (2020) — ZeRO: Memory Optimizations — [arXiv:1910.02054](https://arxiv.org/abs/1910.02054)
- Shoeybi et al. (2019) — Megatron-LM — [arXiv:1909.08053](https://arxiv.org/abs/1909.08053)
- Goyal et al. (2017) — Large Minibatch SGD — [arXiv:1706.02677](https://arxiv.org/abs/1706.02677)
- You et al. (2017) — LARS — [arXiv:1708.03888](https://arxiv.org/abs/1708.03888)
- You et al. (2020) — LAMB — [arXiv:1904.00962](https://arxiv.org/abs/1904.00962)
- Narayanan et al. (2021) — Megatron-LM Pipeline — [arXiv:2104.04473](https://arxiv.org/abs/2104.04473)
- Li et al. (2020) — PyTorch Distributed — [DDP Tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- Zhao et al. (2023) — PyTorch FSDP — [Blog](https://pytorch.org/blog/introducing-pytorch-fully-sharded-data-parallel-api/)

### Blogs & Docs
- [Lilian Weng — Training Large Neural Networks](https://lilianweng.github.io/posts/2021-09-25-train-compute/)
- [HuggingFace — Debugging Mixed Precision](https://huggingface.co/docs/transformers/perf_train_gpu_one#mixed-precision-training)
- [PyTorch AMP docs](https://pytorch.org/docs/stable/amp.html)
- [NVIDIA — Mixed Precision Training](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/)
- [DeepSpeed Documentation](https://www.deepspeed.ai/docs/)
